CuadernoClase-DeterminacionOrbita-JorgeZuluaga.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1LOQh0Md5x6vvRlHWh_ct8BqxnSZICY8i

# Cuaderno de clase
## Mecánica Celeste (2026-1) con Jorge I. Zuluaga
## Determinación de Órbita

In [ ]:
!pip install -Uq pymcel

In [ ]:
import pymcel as pc
import numpy as np
import matplotlib.pyplot as plt
import spiceypy as spy

In [ ]:
tabla, jd, X = pc.consulta_horizons(
    id='Apophis',
    location='@SSB',
    epochs='2026-04-20'
)
X

In [ ]:
rvec = X[:3]
r = np.linalg.norm(rvec)
vvec = X[3:]
v = np.linalg.norm(vvec)
rvec, vvec

Para la determinación de los elementos orbitales necesito estos vectores: $\vec r, \vec v, \vec h, \vec e, \vec n$

In [ ]:
# MARE
hvec = np.cross(rvec, vvec)
h = np.linalg.norm(hvec)

In [ ]:
# Vector de Laplace
mu = pc.constantes.mu_sun  # Aproximacion que no me permiten en la UdeA pero lo voy a hacer..
evec = np.cross(vvec, hvec) / mu - rvec / r
e = np.linalg.norm(evec)

In [ ]:
# Vector de nodos
nvec = np.cross([0,0,1], hvec)
n = np.linalg.norm(nvec)

In [ ]:
hvec, evec , nvec, mu

Ahora a determinar los elementos orbitales $(p, e, I, \Omega, \omega, f)$

In [ ]:
p = h**2 / mu
e = e # Es la magnitu de evec
I = np.arccos(hvec[2]/h)
rad = 180 / np.pi
I*rad

In [ ]:
omegap = np.arccos(nvec@evec/(n*e))
omega = omegap if evec[2] > 0 else 2*np.pi - omegap
omega*rad

In [ ]:
Omegap = np.arccos(nvec@[1,0,0]/n)
if nvec[1] > 0:
  Omega = Omegap
else:
  Omega = 2*np.pi - Omegap

In [ ]:
Omega = Omegap if nvec[1] > 0 else 2*np.pi - Omegap

In [ ]:
Omega*rad

In [ ]:
fp = np.arccos(rvec@evec/(r*e))
f = fp if rvec@vvec/r > 0 else 2*np.pi - fp
f*rad

In [ ]:
def calcular_elementos_orbitales(rvec, vvec, mu):
    r = np.linalg.norm(rvec)
    v = np.linalg.norm(vvec)

    # Momento angular
    hvec = np.cross(rvec, vvec)
    h = np.linalg.norm(hvec)

    # Vector de excentricidad (Laplace)
    evec = np.cross(vvec, hvec) / mu - rvec / r
    e = np.linalg.norm(evec)

    # Vector de nodos
    nvec = np.cross([0, 0, 1], hvec)
    n = np.linalg.norm(nvec)

    # Elementos orbitales
    p = h**2 / mu
    I = np.arccos(hvec[2] / h)

    # Ascensión recta del nodo ascendente (Omega)
    if n == 0:
        Omega = 0 # Órbita ecuatorial, nodo indefinido
    else:
        Omegap = np.arccos(np.dot(nvec, [1, 0, 0]) / n)
        if nvec[1] > 0:
            Omega = Omegap
        else:
            Omega = 2 * np.pi - Omegap

    # Argumento del periapsis (omega)
    if e == 0:
        omegap = 0 # Órbita circular, periapsis indefinida
    elif n == 0:
        omegap = np.arctan2(evec[1], evec[0]) # Órbita ecuatorial, nodo indefinido
    else:
        omegap = np.arccos(np.dot(nvec, evec) / (n * e))
        if evec[2] < 0:
            omegap = 2 * np.pi - omegap

    # Anomalía verdadera (f)
    fp = np.arccos(np.dot(rvec, evec) / (r * e))
    if np.dot(rvec, vvec) > 0:
        f = fp
    else:
        f = 2 * np.pi - fp

    return p, e, I, Omega, omegap, f

In [ ]:
p, e, I, Omega, omega, f = calcular_elementos_orbitales(rvec, vvec, mu)
p, e, I*rad, Omega*rad, omega*rad, f*rad

## Estudio completo de los elementos orbitales de Apophis

In [ ]:
rad = 180 / np.pi
a = p / (1 - e**2)

In [ ]:
print("=" * 57)
print("  Elementos orbitales de Apophis  (2026-04-20, @SSB)")
print("=" * 57)
print(f"  Semilato recto       p  = {p:.6f} AU")
print(f"  Excentricidad        e  = {e:.6f}")
print(f"  Semieje mayor        a  = {a:.6f} AU")
print(f"  Inclinación          I  = {I*rad:.4f}°")
print(f"  Long. nodo asc.      Ω  = {Omega*rad:.4f}°")
print(f"  Arg. periapsis       ω  = {omega*rad:.4f}°")
print(f"  Anomalía verdadera   f  = {f*rad:.4f}°")
print("=" * 57)

Visualización de la órbita de Apophis en el plano eclíptico

In [ ]:
fs_plot = np.linspace(0, 2 * np.pi, 500)
rs_orb = p / (1 + e * np.cos(fs_plot))

In [ ]:
Rz_omega_m   = spy.rotate(omega, 3)
Rx_I_m       = spy.rotate(I, 1)
Rz_Omega_m   = spy.rotate(Omega, 3)
M_perifocal2astro = (Rz_omega_m @ Rx_I_m @ Rz_Omega_m).T

In [ ]:
xfs_orb = rs_orb * np.cos(fs_plot)
yfs_orb = rs_orb * np.sin(fs_plot)
zfs_orb = np.zeros_like(xfs_orb)
rvecs_orb = (M_perifocal2astro @ np.array([xfs_orb, yfs_orb, zfs_orb])).T

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
ax.plot(rvecs_orb[:, 0], rvecs_orb[:, 1], 'b-', label='Órbita de Apophis')
ax.plot(0, 0, 'yo', ms=12, label='Sol (aprox. SSB)')
ax.plot(rvec[0], rvec[1], 'ro', ms=8, label='Apophis 2026-04-20')
ax.set_xlabel('x [AU]')
ax.set_ylabel('y [AU]')
ax.set_title('Órbita de Apophis en el plano eclíptico')
ax.axis('equal')
ax.grid()
ax.legend()
plt.tight_layout()
plt.show()

## Integración de 2 cuerpos con pymcel

Integramos la trayectoria relativa Apophis–Sol usando `pc.doscuerpos_solucion`
y determinamos el momento angular específico $h$ respecto al Sol.

In [ ]:
# ~2 períodos orbitales de Apophis (aprox. 323 días cada uno)
T_apo = 2 * np.pi * np.sqrt(a**3 / mu)
ts_int = np.linspace(0, 2 * T_apo, 2000)

In [ ]:
rs_int, vs_int = pc.doscuerpos_solucion(mu, rvec, vvec, ts_int)

In [ ]:
hvec_int0 = np.cross(rs_int[0], vs_int[0])
h_int0 = np.linalg.norm(hvec_int0)

In [ ]:
print(f"\nh respecto al Sol (integración 2 cuerpos, pymcel):")
print(f"  h = {h_int0:.8f}  (unidades Horizons: AU²/día aprox.)")
print(f"  h analítico (r×v inicial) = {h:.8f}")

## Cuadraturas del problema relativo Sol–Apophis

Las tres integrales primeras (cuadraturas) del problema de dos cuerpos son:

1. **MARE** – Momento Angular Relativo Específico: $h = |\\vec{r}\\times\\dot{\\vec{r}}|$
2. **ERE**  – Energía Relativa Específica: $\\varepsilon = v^2/2 - \\mu/r$
3. **Vector de Laplace** (excentricidad): $\\vec{e}_L = (\\dot{\\vec{r}}\\times\\vec{h})/\\mu - \\hat{r}$

Verificamos que las tres se mantienen constantes a lo largo de la integración.

In [ ]:
# Calcular las tres cuadraturas en cada instante
hvecs_int   = np.cross(rs_int, vs_int)                                   # (N,3)
hs_int      = np.linalg.norm(hvecs_int, axis=1)                          # MARE

In [ ]:
r_norms_int = np.linalg.norm(rs_int, axis=1)
v_norms_int = np.linalg.norm(vs_int, axis=1)
epsilons    = 0.5 * v_norms_int**2 - mu / r_norms_int                    # ERE

In [ ]:
evecs_int   = (np.cross(vs_int, hvecs_int) / mu
               - rs_int / r_norms_int[:, np.newaxis])                    # Vector de Laplace
es_int      = np.linalg.norm(evecs_int, axis=1)

In [ ]:
# Variación relativa (medida de conservación numérica)
delta_h   = (hs_int.max()   - hs_int.min())   / abs(hs_int[0])
delta_eps = (epsilons.max() - epsilons.min())  / abs(epsilons[0])
delta_e   = (es_int.max()   - es_int.min())    / abs(es_int[0])

In [ ]:
print("\nConservación de las cuadraturas a lo largo de la integración:")
print(f"  MARE  h = {hs_int[0]:.8f}  →  Δh/h₀    = {delta_h:.2e}")
print(f"  ERE   ε = {epsilons[0]:.8f}  →  Δε/|ε₀|  = {delta_eps:.2e}")
print(f"  |ê_L| e = {es_int[0]:.8f}  →  Δe/e₀    = {delta_e:.2e}")

In [ ]:
# Gráficas de las cuadraturas vs tiempo
fig, axs = plt.subplots(3, 1, figsize=(10, 9), sharex=True)

In [ ]:
axs[0].plot(ts_int, hs_int, 'b')
axs[0].set_ylabel('MARE  $h$')
axs[0].set_title('Cuadraturas del problema relativo Sol–Apophis')
axs[0].grid()

In [ ]:
axs[1].plot(ts_int, epsilons, 'g')
axs[1].set_ylabel('ERE  $\\varepsilon$')
axs[1].grid()

In [ ]:
axs[2].plot(ts_int, es_int, 'r')
axs[2].set_ylabel('$|\\vec{e}_L|$  (excentricidad)')
axs[2].set_xlabel('Tiempo (unidades Horizons desde 2026-04-20)')
axs[2].grid()

In [ ]:
plt.tight_layout()
plt.show()